# Day 1 — AI System Design, RAG at Scale, Latency & Caching

---

This section is only 2 days. Today we cover **the design content**:

1. **The 5-step framework** every AI system design interview grades you on
2. **Designing RAG at scale** — the most-asked variant
3. **Latency + caching** — the two levers that most improve real systems

Dense day. Read once, then re-read tonight.


## Part 1 — The 5-step system design framework

Interviewer says something vague like *"design a chatbot for our support docs."* You silently unfold this script:

1. **Clarify requirements** (5 min)
2. **Estimate scale** (5 min)
3. **Sketch high-level architecture** (10 min)
4. **Deep-dive one or two components** (15 min)
5. **Discuss tradeoffs, monitoring, cost** (10 min)

**Never skip 1 and 2.** Juniors dive to boxes and arrows. Seniors clarify.


### Step 1 — Clarify

Say aloud: *"Before I sketch anything, can I confirm a few things?"* Then ask:

- **Users** — internal? public? regulated?
- **Task** — search, chat, summarize, extract?
- **Scale** — DAU, queries per day, latency SLO
- **Data** — sources, formats, refresh cadence
- **Must NOT** — PII leaks? hallucinated advice? cost blowups?

Write answers on the whiteboard. Reference them later.


### Step 2 — Back-of-envelope

Numbers first. Interviewers love them.

**Standard shape:**
- DAU → queries per second
- Queries × avg tokens → tokens per day
- Tokens × price → daily cost
- Docs × chunks each → vector count
- Vectors × dim × 4 bytes → storage estimate

**Example** — 10k DAU, 5 questions each:
```
50,000 queries/day ≈ 0.6 QPS avg, ~5 QPS peak
Each query: ~500 in + ~200 out = 700 tokens
Daily: 35 M tokens
At openai/gpt-oss-20b (~$0.10/1M blended): $3.5/day ≈ $105/month
Storage (500k chunks × 384 dim × 4 B) = 768 MB (one box)
```

**Say the numbers out loud.** Don't wait for the interviewer to ask.


### Step 3 — Architecture

Draw and narrate.

```
[Client]
   │
[LB / API Gateway]
   │
[FastAPI / API tier] ─── [Auth]
   │
[Retrieval] ─── [Vector DB] ─── [Object storage for raw docs]
   │
[Reranker]
   │
[LLM (managed or self-hosted)]
   │
[Cache] ─── [Semantic + prompt cache]

Cross-cutting: [Observability]  [Async jobs]  [CI/CD]
```

Walk the request path: *"user hits gateway → auth → API → retrieve → rerank → LLM → response."*


### Step 4 — Deep dive

Interviewer picks one box. Common deep-dives + what you should be ready to say:

| Topic | Say |
|---|---|
| Retrieval | Chunking (500/50 recursive), embedder (MiniLM/BGE/OpenAI), rerank (bge-reranker), hybrid (BM25+dense+RRF) |
| Vector DB | pgvector < 10M vectors, Pinecone above, Chroma for dev |
| Prompt design | System prompt + numbered context + "I don't know" + one-shot example for citations |
| Ingestion | Batch vs incremental (webhook), dedupe by hash, soft-delete via metadata |
| Serving | vLLM self-hosted vs Together AI vs OpenAI, cost/latency table |

For every choice: **the pick**, **the reason**, **one alternative**, and **what would push you to it**.


### Step 5 — Tradeoffs, monitoring, cost

Close every design with these three:

- **Tradeoffs**: "I chose X over Y because [ops / cost / latency]."
- **Monitoring**: "Trace every LLM call in Langfuse. Alert on p95 latency + daily spend."
- **Cost**: "Dominant cost is LLM tokens. If we grow 10×, first move is semantic caching, then a smaller classifier model."

If you close with these three naturally, you'll seem senior.


### Common failure modes

- Jumping to implementation before clarifying → interviewer sighs
- Buzzwords without reason: *"I'd use LangGraph and Pinecone and Kubernetes."* *Why?*
- No numbers ("It'll be fast." → *How fast?*)
- Ignoring cost entirely
- Trying to be "correct" instead of showing tradeoffs


## Part 2 — Designing RAG at scale (1M+ docs, 10k+ users)

Every AI-eng interview asks a variant of *"design a RAG for a million docs and ten thousand users."* Answer key below.


### The reference architecture

```
[Docs / source systems]                          [User]
       │                                            │
  ┌────▼────────┐                             ┌─────▼──────┐
  │ Ingestion   │                             │  Gateway   │
  │ pipeline    │                             └─────┬──────┘
  └────┬────────┘                                   │
       │                                            ▼
       ▼                                    ┌───────────────┐
  ┌───────────────┐    ┌──────────────┐     │  Query API    │
  │ Chunker +     │    │ Metadata DB  │     └───────┬───────┘
  │ embedder      │◄───┤ (Postgres)   │             │
  └────┬──────────┘    └──────────────┘             ▼
       │                                    ┌───────────────┐
       ▼                                    │ Retriever     │
  ┌──────────────┐                          │ (hybrid)      │
  │ Vector DB    │◄─────────────────────────┤               │
  │ (Pinecone /  │                          └───────┬───────┘
  │  Weaviate /  │                                  ▼
  │  pgvector)   │                          ┌───────────────┐
  └──────────────┘                          │ Reranker      │
                                            └───────┬───────┘
                                                    ▼
                                            ┌───────────────┐
                                            │ LLM           │
                                            └───────────────┘
Cross-cutting: [Cache]  [Observability]  [Auth]  [Cost tracker]
```


### The choices, with tradeoffs

**Vector DB pick table:**

| DB | Best at | Watch out |
|---|---|---|
| Pinecone | Managed, easy at any scale | Cost scales linearly |
| Weaviate | Open source + hybrid built in | You run it |
| pgvector | Teams already on Postgres | Slower ANN above ~10M vectors |
| Qdrant | Fast + gRPC | Newer ecosystem |
| Chroma | Dev + small prod | Not battle-tested at 100M+ |

**Fresher script:** *"pgvector until ~10M. Then Pinecone/Weaviate. Switch is ~200 lines because retrieval is abstracted."*

**Retrieval at scale:** BM25 (50 hits) + dense (50 hits) → RRF fusion → cross-encoder rerank → top-5. Reranker on GPU batching ≈ 1 ms per pair.

**Two-LLM pattern:** small model (3B) for routing/classification, big model (70B) for final answer. Interviewers love this as a cost lever.

**Multi-tenant isolation:** metadata filter (default) → namespace-per-tenant (regulated industries).

**Failure modes to name:** query storms during reindex (read replicas), hot-doc problem (cache answers, not just retrieval), slow tail latency (per-call timeouts), cost blowup on viral queries (semantic cache), indirect prompt injection (sanitize retrieved docs before prompt).


### The one-slide 60-second answer

> *"Ingestion: webhook-triggered chunking + embedding, writing to Pinecone with tenant metadata. Retrieval: BM25 + dense + RRF + cross-encoder rerank. Generation: routed to openai/gpt-oss-20b on Together AI, with openai/gpt-oss-20b's smaller sibling for classification. Semantic + prompt caches. Langfuse traces every call. Cost dominated by generation; per-user budget cap + smaller model for cheap paths."*

Rehearse this until it flows.


## Part 3 — Latency & caching

The LLM dominates latency in any RAG:

| Step | Latency |
|---|---|
| Embed query | 20–100 ms |
| Vector DB search | 30–150 ms |
| Rerank | 50–200 ms |
| LLM first token | 200 ms – 2 s |
| LLM output | 30–100 tokens/sec |

**Always-apply latency wins:**
- Stream tokens (first-token latency is what users perceive)
- Parallelize embedding + BM25
- Smaller model for routing / short answers
- Fewer tokens in (shorter system prompt, fewer chunks)
- `max_tokens` cap on output
- Colocate services (LLM in us-east + vector DB in ap-south = pain)


### Three cache layers to know cold

**Layer 1 — Response cache.** Exact query string → cached answer. Hit rate low (<5%) for chat, high for FAQ. Cheap, always add.

**Layer 2 — Semantic cache.** Embed the query, look up in a "cache collection" of past query→answer pairs, return cached answer if similarity > threshold. **20–40% hit rate on support bots — biggest cost saver.** Watch threshold — too loose returns wrong answers.

**Layer 3 — Prompt cache.** OpenAI/Anthropic/Together all support marking a stable system-prompt prefix as cacheable. Provider skips re-processing on repeats. Cuts input-token cost 50–90% for long stable prompts. **Free money.**


In [ ]:
# Sketch of a semantic cache
import numpy as np
from sentence_transformers import SentenceTransformer

class SemanticCache:
    def __init__(self, threshold: float = 0.85):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.threshold = threshold
        self.vecs, self.answers = [], []

    def _embed(self, q):
        v = self.model.encode(q).astype("float32")
        return v / (np.linalg.norm(v) + 1e-9)

    def get(self, q):
        if not self.vecs: return None
        sims = np.stack(self.vecs) @ self._embed(q)
        i = int(sims.argmax())
        return self.answers[i] if sims[i] >= self.threshold else None

    def set(self, q, ans):
        self.vecs.append(self._embed(q)); self.answers.append(ans)

# c = SemanticCache(threshold=0.75)
# c.set("What is the refund policy?", "Refunds within 30 days.")
# print(c.get("How do I get a refund?"))


### Cache invalidation — the hard part

- Response cache → TTL of a few hours
- Semantic cache → **bump a version when your prompt changes** or you'll serve last week's behavior for weeks
- Prompt cache → provider handles it

Build a **`/admin/clear_cache`** on day 1. You'll need it.


### Numbers to memorize for interviews

- Time to first token, big model: **500 ms – 1.5 s**
- Time to first token, small model: **~200 ms**
- Embedding call: **~50 ms**
- Vector DB search on 10M vectors (HNSW): **~30 ms**
- Cross-encoder rerank of 20 pairs: **~100 ms on GPU, ~400 ms on CPU**
- LLM output rate: **30–100 tokens/sec**


## Recap of Day 1

- **5 steps: clarify → estimate → architect → deep-dive → tradeoffs.**
- Always name **numbers**, name **tradeoffs**, close with **monitoring + cost**.
- RAG at scale: **pgvector → Pinecone**, **hybrid + rerank**, **two-LLM pattern**, **metadata-filter tenant isolation**.
- Three caches: **response / semantic / prompt**. Semantic saves the most cost; prompt is free money.
- **Next class:** the 20 interview questions + portfolio + a full mock walkthrough.
